In [ ]:
import copy
import csv
import glob
import json
import librosa
import librosa.display
import random
import os
import sys
import torch
import ast  # for parsing stringified tuples in CSV


import IPython.display as ipd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import wavfile
from scipy.spatial.distance import euclidean
from scipy import stats

sys.path.append("/orcd/data/jhm/001/om2/gelbanna/repos/projects/continuous-speech-recognition")

from front_end.cochlear_model import CochlearModel
from utils import load_yaml_config

sys.path.append("/orcd/data/jhm/001/om2/bjmedina/memory/utils/")
from audio_utils import *


#testing out cochlear model

In [ ]:
# Example instantiation
config = load_yaml_config("/orcd/data/jhm/001/om2/bjmedina/memory/utils/cochleagram_params.yaml")
config['frontend']['cochlear']['filter_params']['sr'] = 44100
model = CochlearModel(config.frontend.cochlear, device="cuda")


# Example forward pass
# x = [torch.randn(1, 1, 16000), torch.randn(1, 1, 8000), torch.randn(1, 1, 32000)]  # Example input tensor
# output = model(x[0].to("cuda"))
# print(output.shape)  # Example output shape 

In [ ]:
def plot_torch_cochleagram(output):
    cochleagram_dB = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.imshow(
        cochleagram_dB,
        origin='lower',
        aspect='auto',
        # extent=[time_bins[0].item(), time_bins[-1].item(),
        #         freq_bins[0].item(), freq_bins[-1].item()],
        cmap='viridis'
    )
    plt.colorbar(label='Amplitude [dB]')
    plt.title("Cochleagram")
    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.tight_layout()
    plt.show()
    
def plot_torch_corrs(output):
    corrs = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.matshow(
        corrs, vmin=0, vmax=1
    )
    plt.colorbar(label='Pearson Correlation')
    plt.title("Segments")
    plt.xlabel("Segments")
    plt.ylabel("Segments")
    plt.tight_layout()
    plt.show()
# plot_torch_cochleagram(output)

In [ ]:
def rms_normalize(signal, target_rms=0.1):
    """
    Normalize a signal to a target RMS level.
    
    Parameters:
        signal (numpy array): The input signal.
        target_rms (float): The desired RMS level (default is 0.1).
    
    Returns:
        numpy array: The RMS-normalized signal.
    """
    rms = np.sqrt(np.mean(signal**2))
    if rms == 0:
        return signal  # Avoid division by zero
    return signal * (target_rms / rms)

In [ ]:
# grabbing stationarity scores for sounds that are in the in the eval set of AudioSet (which is what we typically use for memory experiment)
stationarity_scores_path = "/orcd/data/jhm/001/om2/bjmedina/chexture_choolbox/assets/OVERLAPPED_0.5s_eval_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv"
stationarity_scores_ = pd.read_csv(stationarity_scores_path)

In [ ]:
# grabbing the textures that were selected to be in the memory 
texture_pairs_paths = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exemplar_0.5s_adjacent/texture_filenames.json"

with open(texture_pairs_paths) as f:
    texture_pairs = json.load(f)

In [ ]:
ss_scores_to_texture = defaultdict(list)
times_to_texture     = defaultdict(list)

for texture_pair in texture_pairs:
    texture_id = "/".join(texture_pair['full_path'].split("/")[-2:])

    avg_ss_scores = stationarity_scores_[stationarity_scores_.filepath==texture_id].stationarity_score.tolist()
    times = stationarity_scores_[stationarity_scores_.filepath==texture_id].onset_time.tolist()
    
    ss_scores_to_texture[texture_id] = avg_ss_scores
    times_to_texture[texture_id]     = times

In [ ]:
# plot distribution of stationarity scores
avg_scores = []
for id in ss_scores_to_texture:
    avg_scores.extend(ss_scores_to_texture[id])

In [ ]:
plt.title("Stationarity Scores")
plt.axvline(x=np.mean(avg_scores), label=f"$\mu={np.mean(avg_scores)}$\n$\sigma={np.std(avg_scores)}$")
plt.hist(avg_scores, bins=100, alpha=0.5)
plt.legend()

avg_ss_score = np.mean(avg_scores)
std_ss_score = np.std(avg_scores)

In [ ]:
# consider how you screen for different sound
fps = stationarity_scores_[stationarity_scores_.stationarity_score < avg_ss_score].filepath.tolist(); fps = list(set(fps))

In [ ]:
base_dir = "/om2/data/public/audioset/wavs/eval_segments_downloads/"
offset = 0.1

#save_dir = f"/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/cochleagram_norms_ratio-max_stationarity-lessthan{avg_ss_score:.2f}-offset{offset}/"


In [ ]:
from tqdm.notebook import tqdm
import torchaudio.transforms as T
from IPython.display import Audio

def should_filter_out_yin_cpu_v2(data_np, sr, 
                               min_freq=100, max_freq=3000, 
                               threshold=0.5, std_thresh=0.1, 
                               frame_length=2048//2, hop_length=512//2):
    """
    Detect whether a signal is pure-tone-like based on YIN periodicity estimation.
    CPU version using librosa.yin.
    """
    # Ensure input is a numpy array
    data_np = np.asarray(data_np, dtype=np.float32)

    # Estimate f0 with librosa's YIN
    f0 = librosa.yin(
        y=data_np,
        fmin=min_freq,
        fmax=max_freq,
        sr=sr,
        frame_length=frame_length,
        hop_length=hop_length
    )  # shape: (frames,)

    f0, voiced_flag, voiced_probs = librosa.pyin

    plt.plot(f0)
    plt.ylabel("freq")
    plt.xlabel("time")
    plt.show()
    display(Audio(data_np, rate=sr))

    # Remove invalid f0 estimates (nan when no confident detection)
    #valid_f0 = f0[~np.isnan(f0)]
    valid_f0 = ~np.isnan(f0)

    # Calculate how many frames had a valid f0
    periodicity_score = valid_f0.size / len(f0)

    # Calculate standard deviation of f0 for stability
    if valid_f0.size > 1:
        f0_std = np.std(valid_f0)
    else:
        f0_std = np.inf

    # Decision: periodic enough + stable enough
    is_pure = (periodicity_score > threshold) and (f0_std < std_thresh)

    input()
    plt.clf()

    return is_pure, periodicity_score, f0_std

def should_filter_out_pyin(signal, sr, 
                           min_freq=75, max_freq=450, 
                           voicing_thresh=0.8,
                           periodicity_thresh=0.8,
                           stability_thresh=10.0):
    """
    Detect whether a signal is pure-tone-like using librosa.pyin.
    """
    f0, voiced_flag, voiced_prob = librosa.pyin(signal,
                                                sr=sr,
                                                fmin=min_freq,
                                                fmax=max_freq)

    if f0 is None:
        return False, 0.0, float('inf')  # pyin failed completely

    # Optional: apply soft voicing threshold to filter low-confidence frames
    voiced_mask = voiced_prob > voicing_thresh
    voiced_f0   = f0[voiced_mask]

    periodicity_score = np.sum(voiced_f0) / len(f0)
    f0_std = np.std(voiced_f0) if len(voiced_f0) > 1 else np.inf

    #is_pure = (periodicity_score > periodicity_thresh) and (f0_std < stability_thresh)

    return periodicity_score, f0_std

device = "cuda" if torch.cuda.is_available() else "cpu"

pure_tone_dict = defaultdict(list)
silence_dict   = defaultdict(list)

with tqdm(total=len(fps), file=sys.stdout) as pbar:
    for curr_sound in fps:
        #print(curr_sound)
        audio_path = base_dir + curr_sound
        data, samplerate = librosa.load(audio_path)

        #data = torch.tensor(data, dtype=torch.float32)
        #print(data.shape, samplerate, data.shape[0] / samplerate)

        periodicity_score, f0_std  = should_filter_out_pyin(data, 
                                                            samplerate,
                                                            periodicity_thresh=0.8, 
                                                            stability_thresh=10)#, device=device)
        
        pure_tone_dict[curr_sound] = [periodicity_score, f0_std]

        silence_dur = compute_silence_duration(data, samplerate, threshold=0.01)
        silence_dict[curr_sound] = silence_dur

        pbar.update(1)

In [ ]:
def plot_autocorrelation_peaks(data_np, sr, min_freq=100, max_freq=3000, device="cuda"):
    data = torch.tensor(data_np, dtype=torch.float32, device=device)
    corr = compute_autocorrelation_torch(data).cpu().numpy()

    # Convert frequency range to lags
    min_lag = int(sr / max_freq)
    max_lag = int(sr / min_freq)

    if max_lag >= len(corr):
        print("Signal too short for autocorrelation range.")
        return

    segment = corr[min_lag:max_lag]
    lags = torch.arange(min_lag, max_lag).cpu().numpy()

    # Top 5 peaks
    segment_torch = torch.tensor(segment)
    peak_vals, peak_idxs = torch.topk(segment_torch, k=min(5, len(segment_torch)))

    # Plot full autocorr segment
    plt.figure(figsize=(10, 4))
    plt.plot(lags, segment, label='Autocorrelation')
    plt.scatter(lags[peak_idxs], peak_vals.numpy(), color='red', label='Top Peaks')

    # Annotate std and first peak
    peak_std = torch.std(peak_vals).item()
    first_peak = peak_vals[0].item()
    plt.title(f"First Peak: {first_peak:.3f} | Std of Peaks: {peak_std:.3f}")
    plt.xlabel("Lag (samples)")
    plt.ylim([-1,1])
    plt.ylabel("Normalized Autocorr")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
audio_path = base_dir + "eval_segments_downloads_0/YT_dt8eRF7lCt4.wav"
data, samplerate = librosa.load(audio_path)

plot_autocorrelation_peaks(data, samplerate, min_freq=100, max_freq=3000, device="cuda")

In [ ]:
audio_paths = glob.glob("/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/max_cochleagram_mse-time_avg_mse-25.0db-lowpassfiltered-10hz-lessthan-0.62-offset0.1-segment_duration0.5/all_pairs/*dt8eRF7lCt4*")
data, samplerate = librosa.load(audio_paths[0])
print(audio_paths[0])
plot_autocorrelation_peaks(data[len(data)//2:], samplerate, min_freq=100, max_freq=3000, device="cuda")
plt.plot(data[len(data)//2:], alpha=0.5)

In [ ]:
data, samplerate = librosa.load(audio_paths[1])
print(audio_paths[1])
plot_autocorrelation_peaks(data[:len(data)//2], samplerate, min_freq=100, max_freq=3000, device="cuda")
plt.plot(data[:len(data)//2], alpha=0.5)

In [ ]:
audio_paths = glob.glob("/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/max_cochleagram_mse-time_avg_mse-25.0db-lowpassfiltered-10hz-lessthan-0.62-offset0.1-segment_duration0.5/all_pairs/*dt8eRF7lCt4*")
data, samplerate = librosa.load(audio_paths[0])
plot_autocorrelation_peaks(data, samplerate, min_freq=100, max_freq=3000, device="cuda")

In [ ]:
plt.plot(data)

In [ ]:
import math
# Filter out entries with NaNs in val1
pure_tone_dict_no_nan = {
    k: v for k, v in pure_tone_dict.items()
    if not math.isnan(v[0])
}

# Now sort descending by val1
pure_tone_dict_sorted = dict(
    sorted(pure_tone_dict_no_nan.items(), key=lambda item: -float(item[1][0]))
)

#pure_tone_dict_sorted = dict(sorted(pure_tone_dict.items(), key=lambda item: -float(item[1][0])))

silence_dict_sorted = dict(sorted(silence_dict.items(), key=lambda item: item[1]))

In [ ]:
# Filter out entries where val1 is 0, nan, or inf
nonzero_clips = {
    k: v for k, v in pure_tone_dict.items()
    if v[0] > 0 and not (math.isnan(v[0]) or math.isinf(v[0]))
}

# Sort by ascending val1 (most pure-tone-like to less)
nonzero_sorted = dict(sorted(nonzero_clips.items(), key=lambda item: item[1][0]))

# Listen to first 10 nonzero clips
print("🔊 First 10 clips with smallest non-zero pure-tone-like ratios:")
j = 380
offset = 2
for path, prob in list(nonzero_sorted.items())[j:j+offset]:
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate:.2f}s")
    display(Audio(audio_path))

In [ ]:
for k, v in pure_tone_dict_sorted.items():
    print(k, v[0], type(v))

In [ ]:
from IPython.display import Audio, display
num_clips = len(pure_tone_dict_sorted)
# Top 10 most pure-tone-like
print("🔊 Top 10 clips with most pure-tone-like clips:")
for path, prob in list(pure_tone_dict_sorted.items())[:10]:  # reverse so highest first
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))


In [ ]:
save_dir = f"/orcd/data/jhm/001/om2/bjmedina/data/filters/pure_tone_like_yin"
import os
import soundfile as sf

base_save_dir = "/orcd/data/jhm/001/om2/bjmedina/data/filters/pure_tone_like_yin/saved_clips"
categories = ["top", "middle", "bottom"]

# Make the folders if they don't exist
for category in categories:
    os.makedirs(os.path.join(base_save_dir, category), exist_ok=True)

num_clips = len(pure_tone_dict_sorted)

def save_and_display_clip(category, index, path, prob):
    print(f"{path} | ratio: {prob}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    duration = len(data) / samplerate
    print(f"Total length of clip: {duration:.2f}(s)")

    # Save to category-specific subfolder
    filename = f"{index:02d}_{os.path.basename(path)}"
    save_path = os.path.join(base_save_dir, category, filename)
    sf.write(save_path, data, samplerate)

    # Display in notebook
    display(Audio(audio_path))


# Top 10 most pure-tone-like
print("🔊 Top 10 clips with most pure-tone-like content:")
for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[-10:][::-1]):
    save_and_display_clip("top", i, path, prob)

# # Middle 10
# print("\n🔊 Middle 10 clips:")
# mid_start = num_clips // 2 - 5
# mid_end = num_clips // 2 + 5
# for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[mid_start:mid_end]):
#     save_and_display_clip("middle", i, path, prob)

# Bottom 10 least pure-tone-like
print("\n🔊 Bottom 10 clips (least pure-tone-like):")
for i, (path, prob) in enumerate(list(pure_tone_dict_sorted.items())[:10]):
    save_and_display_clip("bottom", i, path, prob)

In [ ]:
from IPython.display import Audio, display
num_clips = len(silence_dict_sorted)
# Top 10 most voice-like
print("🔊 Top 10 clips with most silent clips:")
for path, s_dur in list(silence_dict_sorted.items())[-10:][::-1]:  # reverse so highest first
    print(f"{path} | silence (s): {s_dur}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))


# Bottom 10 least voice-like
print("🔊 Bottom 10 clips with least silent clips:")
for path, s_dur in list(silence_dict_sorted.items())[:10]: 
    print(f"{path} | silence (s): {s_dur}")
    audio_path = base_dir + path
    data, samplerate = librosa.load(audio_path)
    print(f"Total length of clip: {len(data)/samplerate}(s)")
    display(Audio(audio_path))